# Trading Agent — Year Analysis

Runs all 22 strategies on every trading day of the selected year, sequentially.
Simulates recommendations, records outcomes, updates adaptive weights.

**Learning phase (2016–2022):** weights adapt every 20 trading days  
**Testing phase (2023–2026):** weights frozen from end of 2022

Safe to interrupt — resumes from last completed day on next run.

In [12]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

from config.settings import get_phase, LEARNING_END_YEAR
from backtester.engine import run_year
from strategies import ALL_STRATEGIES

print(f'22 strategies loaded: {[s.name for s in ALL_STRATEGIES]}')

22 strategies loaded: ['ORB-15', 'ORB-30', 'PDH-PDL', 'GAP-CONT', 'VOL-SPIKE', 'VWAP-REV', 'RSI-EXT', 'BOLLINGER', 'GAP-FADE', 'EMA-CROSS', 'SUPERTREND', 'MACD', 'SR-BREAK', 'NR7', 'FIRST-CANDLE', 'CPR', 'CAMARILLA', 'ADX-FILTER', 'VWAP-STDDEV', 'STOCHASTIC', 'VPOC', 'REL-STR']


## Check current progress

In [13]:
import json
from config.settings import PROGRESS_FILE, WEIGHTS_FILE

# Show what has been downloaded and what has been analysed
progress = json.loads(PROGRESS_FILE.read_text()) if PROGRESS_FILE.exists() else {}
weights  = json.loads(WEIGHTS_FILE.read_text()) if WEIGHTS_FILE.exists() else {}

print('DOWNLOAD STATUS:')
for yr, data in sorted(progress.get('download', {}).items()):
    done = len(data.get('completed', []))
    fail = len(data.get('failed', []))
    print(f'  {yr}: {done} stocks downloaded, {fail} failed [{data.get("status","?")}]')

print()
print('ANALYSIS STATUS:')
if not progress.get('analysis'):
    print('  No analysis done yet — ready to start 2016')
else:
    for yr, data in sorted(progress.get('analysis', {}).items()):
        phase = get_phase(int(yr))
        print(f'  {yr} [{phase}]: last processed = {data.get("last_processed_date","none")}')

print()
print('CURRENT WEIGHTS (top 5 by value):')
if weights:
    for k, v in sorted(weights.items(), key=lambda x: -x[1])[:5]:
        print(f'  {k:<15} {v:.3f}')
else:
    print('  All strategies at default weight 1.0 (first run)')

DOWNLOAD STATUS:
  2016: 334 stocks downloaded, 166 failed [partial]
  2017: 350 stocks downloaded, 150 failed [partial]

ANALYSIS STATUS:
  2016 [LEARNING]: last processed = 2016-12-30

CURRENT WEIGHTS (top 5 by value):
  VOL-SPIKE       3.000
  NR7             1.000
  ORB-15          0.500
  ORB-30          0.500
  PDH-PDL         0.500


## Run Analysis

Set `YEAR_TO_ANALYSE` and run the cell.  
The engine processes every trading day in order — safe to stop and resume.

In [14]:
# ── SET THIS ──
YEAR_TO_ANALYSE = 2017
# ──────────────

phase = get_phase(YEAR_TO_ANALYSE)
print(f'Year   : {YEAR_TO_ANALYSE}')
print(f'Phase  : {phase}')
print(f'Weights: {"FROZEN (testing mode)" if phase == "TESTING" else "adaptive — will update every 20 days"}')
print()

if phase == 'TESTING' and YEAR_TO_ANALYSE == 2023:
    print('NOTE: Testing phase uses weights frozen at end of 2022 learning.')

Year   : 2017
Phase  : LEARNING
Weights: adaptive — will update every 20 days



In [15]:
# Run the year — this processes all trading days sequentially
# Shows a progress bar, logs weight updates, saves checkpoint after each day

summary = run_year(year=YEAR_TO_ANALYSE)

print()
print('=' * 50)
print(f'YEAR {YEAR_TO_ANALYSE} COMPLETE')
print('=' * 50)
print(f'Total trades  : {summary["total_trades"]}')
print(f'Win rate      : {summary["win_rate"]}%')
print(f'Total P&L     : Rs {summary["total_pnl"]:,.0f}')
print()
print('Top 5 strategies by final weight:')
for k, v in sorted(summary['final_weights'].items(), key=lambda x: -x[1])[:5]:
    print(f'  {k:<15} {v:.3f}')

23:38:05 INFO Starting 2017 [LEARNING] — weights adaptive
23:38:05 INFO Loading stock data into memory...


23:38:13 INFO Loaded 350 stocks for 2017
Analysing 2017:   8%|▊         | 19/248 [10:55<2:20:24, 36.79s/day]23:49:47 INFO   SUPERTREND      0.50 -> 0.10  (suppress, wr=30%)
23:49:47 INFO   ADX-FILTER      0.50 -> 0.25  (reduce, wr=40%)
23:49:47 INFO   STOCHASTIC      0.50 -> 0.10  (suppress, wr=20%)
23:49:47 INFO   VWAP-REV        0.50 -> 0.75  (boost, wr=70%)
23:49:47 INFO   CPR             0.50 -> 0.25  (reduce, wr=40%)
23:49:47 INFO   VWAP-STDDEV     0.50 -> 0.10  (suppress, wr=20%)
23:49:47 INFO   EMA-CROSS       0.50 -> 0.10  (suppress, wr=20%)
23:49:47 INFO   GAP-CONT        0.50 -> 0.75  (boost, wr=83%)
23:49:47 INFO Weights updated after 2017-01-30
Analysing 2017:  16%|█▌        | 39/248 [10:45:32<158:19:03, 2727.00s/day] 10:24:28 INFO   ORB-15          0.50 -> 0.10  (suppress, wr=20%)
10:24:28 INFO   ORB-30          0.50 -> 0.10  (suppress, wr=20%)
10:24:28 INFO   PDH-PDL         0.50 -> 0.10  (suppress, wr=20%)
10:24:28 INFO   RSI-EXT         0.50 -> 0.10  (suppress, wr=20%)



YEAR 2017 COMPLETE
Total trades  : 736
Win rate      : 40.6%
Total P&L     : Rs 1,852,033

Top 5 strategies by final weight:
  VOL-SPIKE       1.500
  NR7             1.000
  ORB-15          0.500
  PDH-PDL         0.500
  CPR             0.500


## Review Results

In [16]:
# View strategy weights after analysis
import pandas as pd
from config.settings import WEIGHTS_FILE

weights = json.loads(WEIGHTS_FILE.read_text()) if WEIGHTS_FILE.exists() else {}
w_df = pd.DataFrame(weights.items(), columns=['Strategy', 'Weight'])
w_df = w_df.sort_values('Weight', ascending=False).reset_index(drop=True)

print(f'Strategy weights after {YEAR_TO_ANALYSE}:')
print(w_df.to_string(index=False))

Strategy weights after 2017:
    Strategy  Weight
   VOL-SPIKE  1.5000
         NR7  1.0000
     PDH-PDL  0.5000
      ORB-15  0.5000
         CPR  0.5000
    VWAP-REV  0.2500
    GAP-FADE  0.1265
      ORB-30  0.1250
    GAP-CONT  0.1000
   BOLLINGER  0.1000
   EMA-CROSS  0.1000
  SUPERTREND  0.1000
        MACD  0.1000
     RSI-EXT  0.1000
    SR-BREAK  0.1000
FIRST-CANDLE  0.1000
   CAMARILLA  0.1000
  ADX-FILTER  0.1000
 VWAP-STDDEV  0.1000
  STOCHASTIC  0.1000
        VPOC  0.1000
     REL-STR  0.1000


In [20]:
# View memory file written by the engine
from config.settings import MEMORY_DIR

memory_file = MEMORY_DIR / 'strategy_agent.md'
if memory_file.exists():
    print(memory_file.read_text(encoding='utf-8'))
else:
    print('Memory file not yet written.')

# Trading Agent — Strategy Memory

Records year-by-year strategy performance and learned weights.
Per-strategy win rates tracked from 2017 onwards (2016 performance data was not persisted).

---

## Year 2016 Summary
- Total trades : 693  (238 trading days, max 3/day)
- Win rate     : 30.2%
- Total P&L    : Rs 10,11,970

### Strategy Weights after 2016
| Strategy        | Weight | Verdict     | Note                                      |
|-----------------|--------|-------------|-------------------------------------------|
| VOL-SPIKE       |   3.00 | BEST        | >60% win rate — volume at resistance works in 2016 |
| NR7             |   1.00 | NEUTRAL     | 40–60% win rate or too few signals        |
| GAP-FADE        |   0.10 | SUPPRESSED  | <30% win rate — fading gaps burned in 2016 trend |
| All others (19) |   0.50 | REDUCED     | 30–40% win rate — strategies active but underperforming |

**Key insight:** Only VOL-SPIKE crossed the 60% win-rate threshold in 2016.
GAP-FADE hit min

In [21]:
# Plot strategy weights as bar chart
import plotly.graph_objects as go

weights = json.loads(WEIGHTS_FILE.read_text()) if WEIGHTS_FILE.exists() else {}
sorted_w = sorted(weights.items(), key=lambda x: -x[1])
names, vals = zip(*sorted_w) if sorted_w else ([], [])

fig = go.Figure(go.Bar(
    x=names, y=vals,
    marker_color=['green' if v >= 1.0 else 'orange' if v >= 0.5 else 'red' for v in vals]
))
fig.add_hline(y=1.0, line_dash='dash', line_color='white', annotation_text='Default=1.0')
fig.update_layout(
    title=f'Strategy Weights after {YEAR_TO_ANALYSE}',
    xaxis_title='Strategy', yaxis_title='Weight',
    template='plotly_dark', height=400
)
fig.show()

In [22]:
# Next session prompt — what to type next time you open Claude VSCode
progress = json.loads(PROGRESS_FILE.read_text()) if PROGRESS_FILE.exists() else {}
analysis = progress.get('analysis', {})

done_years = sorted(int(y) for y in analysis.keys())
next_year  = max(done_years) + 1 if done_years else 2016
next_phase = get_phase(next_year)

print('=' * 60)
print('NEXT SESSION')
print('=' * 60)
print(f'Next year to process: {next_year} [{next_phase}]')
print()
print('Steps for next session:')
print(f'  1. Open notebook 01_setup_and_download.ipynb')
print(f'     Set YEAR_TO_DOWNLOAD = {next_year} and run Section 4')
print(f'  2. Open this notebook (02_analyse_year.ipynb)')
print(f'     Set YEAR_TO_ANALYSE = {next_year} and run Section Run Analysis')
print()
print('Or type in Claude VSCode: "Continue trading agent analysis from checkpoint"')

NEXT SESSION
Next year to process: 2018 [LEARNING]

Steps for next session:
  1. Open notebook 01_setup_and_download.ipynb
     Set YEAR_TO_DOWNLOAD = 2018 and run Section 4
  2. Open this notebook (02_analyse_year.ipynb)
     Set YEAR_TO_ANALYSE = 2018 and run Section Run Analysis

Or type in Claude VSCode: "Continue trading agent analysis from checkpoint"
